In [1]:
# run once to download dependencies
# %pip install pyarrow scikit-learn

In [2]:
# import all libraries
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow
import pyarrow.compute
from sklearn.model_selection import StratifiedGroupKFold

# load environment variables (mayb not be needed if running in a notebook)
import os
from dotenv import load_dotenv

In [3]:
# run once to convert the raw data to parquet format

# USER_FILE = DATA_DIR / "user_behavior_data.txt"
# PRODUCT_FILE = DATA_DIR / "product_meta_data.txt"

load_dotenv()

DATA_DIR = Path(os.getenv("DATA_DIR"))
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR} (check your .env)"

USER_RAW = DATA_DIR / "user_behavior_raw.parquet"
PRODUCT_RAW = DATA_DIR / "product_metadata_raw.parquet"

# pd.read_csv(USER_FILE, sep="	", header=0, dtype=str, na_filter=False).to_parquet(USER_RAW, index=False)
# pd.read_csv(PRODUCT_FILE, sep="	", header=0, dtype=str, na_filter=False).to_parquet(PRODUCT_RAW, index=False)


In [4]:
print([str(p) for p in DATA_DIR.parents[2].rglob("*.parquet")])

['C:\\Users\\tokqu\\OneDrive - National University of Singapore\\Y4S1\\BT4222\\Source Code\\JDsearch\\JDsearch\\product_metadata_clean.parquet', 'C:\\Users\\tokqu\\OneDrive - National University of Singapore\\Y4S1\\BT4222\\Source Code\\JDsearch\\JDsearch\\product_metadata_raw.parquet', 'C:\\Users\\tokqu\\OneDrive - National University of Singapore\\Y4S1\\BT4222\\Source Code\\JDsearch\\JDsearch\\user_behavior_clean.parquet', 'C:\\Users\\tokqu\\OneDrive - National University of Singapore\\Y4S1\\BT4222\\Source Code\\JDsearch\\JDsearch\\user_behavior_raw.parquet']


### Step 1a: Data Preprocessing

- Step 1: Load parquet files
- Step 2: Assign `slate_id`
- Step 3: Parse separators (`_` lists, `\x18` query/name terms)
- Step 4: Validate values
- Step 5: Fix data types (text/float → int)
- Step 6: Check list lengths
- Step 7: Flag query-less history actions (`has_query`)
- Step 8: Compute days before search
- Step 9: Flag missing metadata (`has_metadata`)
- Step 10: Shuffle candidates within each slate
- Step 11: Save cleaned data

In [5]:
# Step 1: run to load the parquet files 
user_df = pd.read_parquet(USER_RAW)
product_df = pd.read_parquet(PRODUCT_RAW)


In [7]:
# Step 2: assign a unique slate_id to each row
user_df.insert(0, "slate_id", np.arange(len(user_df), dtype=np.int32))

In [8]:
# Step 3: parse separators 
def split_lists(df, cols, sep="_"):
    # Split each column's text into a list: "101_202_303" -> ["101", "202", "303"]
    for col in cols:
        df[col] = df[col].astype("large_string[pyarrow]").str.split(sep)
    return df

LIST_COLS = ["candidate_wid_list", "candidate_label_list",
             "history_qry_list", "history_wid_list",
             "history_type_list", "history_time_list"]

user_df = split_lists(user_df, LIST_COLS)               
user_df = split_lists(user_df, ["query"], sep="\x18")    
product_df = split_lists(product_df, ["name"], sep="\x18") 

In [9]:
# Step 4: validate allowed values
labels = user_df["candidate_label_list"].list.flatten()
types = user_df["history_type_list"].list.flatten()
times = user_df["history_time_list"].list.flatten()

print("Labels:", labels.unique())              
print("Types:", types.unique())               
print("Invalid time gaps:", (~times.str.isdigit()).sum())  

Labels: <ArrowExtensionArray>
['1.0', '0.0', '2.0', '3.0']
Length: 4, dtype: large_string[pyarrow]
Types: <ArrowExtensionArray>
['ORD', 'CART', 'CLICK', 'FLW']
Length: 4, dtype: large_string[pyarrow]
Invalid time gaps: 0


In [10]:
# Step 5: convert text/float to integers
def to_int(df, cols, is_list=True):
    # Convert columns from text to integers. Lists of text -> lists of integers.
    for col in cols:
        if is_list:
            df[col] = (df[col]
                       .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.float64())))
                       .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.int64()))))
        else:
            df[col] = df[col].astype("int64")
    return df

user_df = to_int(user_df, ["candidate_wid_list", "candidate_label_list",
                           "history_wid_list", "history_time_list"])

product_df = to_int(product_df, ["wid", "brand_id", "cate_id_1", "cate_id_2",
                                 "cate_id_3", "cate_id_4", "shop_id"], is_list=False)

In [11]:
# Step 6: check list lengths match within each row
cand_len = user_df["candidate_wid_list"].list.len()
label_len = user_df["candidate_label_list"].list.len()
print("Candidate/label mismatches:", (cand_len != label_len).sum())   # expect 0

hist_len = user_df["history_wid_list"].list.len()
for col in ["history_qry_list", "history_type_list"]:
    print(col, "mismatches:", (user_df[col].list.len() != hist_len).sum())   # expect 0

# history_time_list has one extra value: the gap to the current search (first value is always 0)
print("history_time_list mismatches:",
      (user_df["history_time_list"].list.len() != hist_len + 1).sum())   # expect 0

Candidate/label mismatches: 0
history_qry_list mismatches: 0
history_type_list mismatches: 0
history_time_list mismatches: 0


In [12]:
# Step 7: flag whether each past action came from a search ("-1" = no query)

def add_has_query(df):
    # True/False per past action: did it come from a search?
    arr = pyarrow.array(df["history_qry_list"])
    flags = pyarrow.compute.not_equal(arr.flatten(), "-1")
    out = pyarrow.ListArray.from_arrays(pyarrow.compute.subtract(arr.offsets, arr.offsets[0]), flags)
    df["history_has_query"] = pd.Series(out, index=df.index, dtype=pd.ArrowDtype(out.type))
    return df

user_df = add_has_query(user_df)

In [13]:
# Step 8: compute days before the search
def days_before(times):
    # [0, 86400, 3600, 60] -> [1.04, 0.04, 0.0007] days before the search
    t = np.asarray(times)
    return np.cumsum(t[::-1])[::-1][1:] / 86400

user_df["history_days_before"] = (user_df["history_time_list"]
                                  .apply(days_before)
                                  .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.float64()))))

In [14]:
# Step 9: add has_metadata flags
def add_has_metadata(df, wid_col, new_col, known_wids):
    # True/False per product in a list column: is it in the product metadata?
    arr = pyarrow.array(df[wid_col])
    flags = pyarrow.array(pd.Series(arr.flatten()).isin(known_wids).to_numpy())
    out = pyarrow.ListArray.from_arrays(pyarrow.compute.subtract(arr.offsets, arr.offsets[0]), flags)
    df[new_col] = pd.Series(out, index=df.index, dtype=pd.ArrowDtype(out.type))
    return df

user_df = add_has_metadata(user_df, "candidate_wid_list", "candidate_has_metadata", product_df["wid"])
user_df = add_has_metadata(user_df, "history_wid_list", "history_has_metadata", product_df["wid"])

In [15]:
# Step 10: shuffle candidates within each slate
SEED = 42

def shuffle_slates(df, cols, seed=SEED):
    # Shuffle each row's candidates, moving all listed columns in the same order.
    rng = np.random.default_rng(seed)
    perms = [rng.permutation(n) for n in df[cols[0]].list.len()]
    for col in cols:
        dtype = df[col].dtype
        df[col] = pd.Series([np.asarray(x)[p] for x, p in zip(df[col], perms)], index=df.index).astype(dtype)
    return df

user_df = shuffle_slates(user_df, ["candidate_wid_list", "candidate_label_list", "candidate_has_metadata"])

In [ ]:
# Save the cleaned data to parquet files
# user_df.to_parquet(DATA_DIR / "user_behavior_clean.parquet", index=False)

# PRODUCT_COLS = ["wid", "name", "brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "shop_id"]
# product_df[PRODUCT_COLS].to_parquet(DATA_DIR / "product_metadata_clean.parquet", index=False)

### Step 1b: Train/Val/Test Splitting

Stratified, grouped 80/10/10 split
- Stratified on: `has_purchase`, history-length bucket, slate-length bucket
- Grouped on: `query` string (rows sharing a query always land in the same split)
- Done by cutting the data into 10 folds: 8 -> train, 1 -> val, 1 -> test

Input to `split_data`: cleaned slate-level dataframe (one row per search/user).
Required columns:
`slate_id`, `query` (string), `candidate_wid_list`, `candidate_label_list` (arrays of equal length), `history_wid_list` (array)

In [21]:
# split into train/test/val
SEED = 42

def add_split_helper_cols(df, hist_bins=(0, 29, 78, 188, np.inf), slate_bins=(0, 76, 90, 95, np.inf)):
    # Adds has_purchase, hist_len, slate_len, query_key and the combined stratum label
    out = df.copy()
    out["query_key"] = out["query"].apply(" ".join)   # ["12", "845"] -> "12 845", so it can be grouped
    out["has_purchase"] = out["candidate_label_list"].apply(lambda x: int((np.asarray(x) == 3).any()))
    out["hist_len"] = out["history_wid_list"].apply(len)
    out["slate_len"] = out["candidate_wid_list"].apply(len)
    hist_b = pd.cut(out["hist_len"], bins=list(hist_bins), labels=False, include_lowest=True)
    slate_b = pd.cut(out["slate_len"], bins=list(slate_bins), labels=False, include_lowest=True)
    out["stratum"] = (out["has_purchase"].astype(str) + "_" + hist_b.astype(str) + "_" + slate_b.astype(str))
    return out


def split_data(df, seed=SEED, n_folds=10, n_val_folds=1, n_test_folds=1):
    # Returns a copy of df with a 'split' column in {'train','val','test'}
    # 80/10/10 with the defaults. Rows with the same query stay in the same split.
    out = add_split_helper_cols(df)
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    fold = np.full(len(out), -1, dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(out, y=out["stratum"], groups=out["query_key"])):
        fold[idx] = k
    assert (fold >= 0).all()

    split = np.where(fold < n_folds - n_val_folds - n_test_folds, "train",
             np.where(fold < n_folds - n_test_folds, "val", "test"))
    out["split"] = split
    return out


def check_split(df):
    # Prints proportions and sanity checks.
    print("Rows per split:\n", df["split"].value_counts(normalize=True).round(4), "\n")
    summary = df.groupby("split").agg(
        rows=("slate_id", "size"),
        purchase_rate=("has_purchase", "mean"),
        mean_hist_len=("hist_len", "mean"),
        median_hist_len=("hist_len", "median"),
        mean_slate_len=("slate_len", "mean"),
    )
    print(summary.round(4), "\n")
    # no query may appear in more than one split
    n_splits_per_query = df.groupby("query_key")["split"].nunique()
    assert (n_splits_per_query == 1).all(), "Leak: a query appears in multiple splits"
    print("OK: every query is in exactly one split")
    n_eval = df[(df["split"] != "train") & (df["has_purchase"] == 1)].groupby("split").size()
    print("Purchase slates usable for evaluation:\n", n_eval.to_string())


def save_split_ids(df, path="split_ids.parquet"):
    df[["slate_id", "split"]].to_parquet(path, index=False)

In [22]:
df = split_data(user_df)
check_split(df)
save_split_ids(df)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)    # full candidate lists, already shuffled
test_df  = df[df["split"] == "test"].reset_index(drop=True)   # full candidate lists, already shuffled

Rows per split:
 split
train    0.8
val      0.1
test     0.1
Name: proportion, dtype: float64 

         rows  purchase_rate  mean_hist_len  median_hist_len  mean_slate_len
split                                                                       
test    17383         0.1168       152.6208             78.0         89.1644
train  139065         0.1168       153.4908             78.0         89.2170
val     17383         0.1168       153.5443             78.0         89.3460 

OK: every query is in exactly one split
Purchase slates usable for evaluation:
 split
test    2030
val     2030


### Step 1c: Negative sampling
`sample_negatives` takes the TRAIN split only.

For every training slate:
- keep ALL items with label 1/2/3 (positives)
- keep up to `n_neg` (defalut 20) randomly chosen label-0 items (negatives)
- slates with no positives go through the same rule (up to `n_neg` label-0 items)

Output is one row per (slate, candidate) with columns:
`slate_id`, `wid`, `label`, `y`, `sample_weight`

`y`=1 only for label 3, weights click 1, cart 5, purchase 10

Call it again every epoch with a new `epoch` number to re-sample negatives.


In [23]:
WEIGHTS = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}

def sample_negatives(train_df, n_neg=20, epoch=0, seed=SEED, weights=WEIGHTS):
    rng = np.random.default_rng(seed + epoch)
    slate_ids, wids, labels = [], [], []

    for sid, w, l in zip(train_df["slate_id"].values,
                         train_df["candidate_wid_list"].values,
                         train_df["candidate_label_list"].values):
        w = np.asarray(w)
        l = np.asarray(l).astype(int)
        pos_idx = np.flatnonzero(l > 0)
        neg_idx = np.flatnonzero(l == 0)
        if len(neg_idx) > n_neg:
            neg_idx = rng.choice(neg_idx, size=n_neg, replace=False)
        keep = np.concatenate([pos_idx, neg_idx])
        slate_ids.append(np.full(len(keep), sid))
        wids.append(w[keep])
        labels.append(l[keep])

    out = pd.DataFrame({
        "slate_id": np.concatenate(slate_ids),
        "wid": np.concatenate(wids),
        "label": np.concatenate(labels).astype(np.int8),
    })

    out["sample_weight"] = out["label"].map(weights).astype(np.float32)
    out["y"] = (out["label"] == 3).astype(np.int8)

    # shuffle so rows of one slate are not stored positives-first
    return out.sample(frac=1.0, random_state=seed + epoch).reset_index(drop=True)


def check_sampling(train_df, sampled, n_neg=20):
    per_slate = sampled.groupby("slate_id").size()
    print("Rows after sampling:", len(sampled), "| slates:", sampled["slate_id"].nunique(),
          "of", len(train_df))
    print("Items per slate (min/median/max):", per_slate.min(), per_slate.median(), per_slate.max())
    print("Label counts:\n", sampled["label"].value_counts().sort_index().to_string())
    neg_per_slate = sampled[sampled["label"] == 0].groupby("slate_id").size()
    assert neg_per_slate.max() <= n_neg
    print("At most", n_neg, "label-0 items per slate")

In [24]:
sampled = sample_negatives(train_df, n_neg=20, epoch=0)
check_sampling(train_df, sampled)

# Per-epoch loop for the neural model (Model 2):
# Each epoch sees a different set of about 20 label-0 items per slate
# for epoch in range(num_epochs):
#     sampled = sample_negatives(train_df, n_neg=20, epoch=epoch)
#     ... build features / DataLoader from `sampled`, train one epoch ...

# For Model 1, sample once (epoch=0) and save: sampled.to_parquet("train_sampled.parquet")
# Evaluation: use val_df / test_df candidate lists directly, no sampling.

Rows after sampling: 2970087 | slates: 139065 of 139065
Items per slate (min/median/max): 1 21.0 69
Label counts:
 label
0    2713285
1     179639
2      60488
3      16675
At most 20 label-0 items per slate
